# 학습목표
- Iceberg에서 스키마 진화(schema evolution)의 개념을 이해한다.
- 컬럼 추가(Add), 삭제(Drop), 이름 변경(Rename)을 통해 메타데이터만 수정되고 데이터 파일은 그대로 유지됨을 배운다.
- Iceberg가 불변 Field ID를 사용하여 데이터 무결성을 보장하는 원리를 이해한다.
- 다른 테이블 포맷 대비 Iceberg가 스키마 변경에 얼마나 강력하고 효율적인지 실습으로 익힌다.

---

# 📌 TL;DR
- Schema evolution = 테이블 구조 변경(컬럼 추가/삭제/변경) 시 데이터 파일은 건드리지 않고 메타데이터만 수정하는 기능  
- Iceberg는 **Field ID**를 통해 컬럼 이름이 바뀌어도 기존 데이터와 정확히 매핑됨  
- ADD / DROP / RENAME 같은 작업에서도 Parquet 파일은 재작성되지 않아 성능과 안정성이 높음  
- 결과적으로 Iceberg는 **대규모 데이터 환경에서 유연하고 안전한 스키마 변경**을 가능하게 함  

---

In [5]:
import json
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from IPython.display import Image
from utils.get_iceberg_metadata import get_metadata, print_avro_item
from utils.iceberg_metadata_properties import IcebergMetadataJsonField, FIELD_DESCRIPTIONS

---

In [6]:
# spark app name 지정을 위해 spark session stop
spark.stop()

In [7]:
spark = SparkSession.builder \
    .appName("Iceberg-schema-evoluition") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.sql.catalog.my_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.my_catalog.type", "rest") \
    .config("spark.sql.catalog.my_catalog.uri", "http://iceberg-rest:8181") \
    .config("spark.sql.catalog.my_catalog.warehouse", "s3a://warehouse/") \
    .config("spark.sql.catalog.my_catalog.io-impl", "org.apache.iceberg.aws.s3.S3FileIO") \
    .config("spark.sql.catalog.my_catalog.s3.endpoint", "http://minio:9000") \
    .config("spark.sql.catalog.my_catalog.s3.access-key-id", "admin") \
    .config("spark.sql.catalog.my_catalog.s3.secret-access-key", "password") \
    .config("spark.sql.catalog.my_catalog.s3.path-style-access", "true") \
    .getOrCreate()

25/10/12 06:53:13 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [12]:
catalog = "my_catalog"
database = "my_database"
table = "customer_order"
table_full_name = f"{catalog}.{database}.{table}"
table_full_name

'my_catalog.my_database.customer_order'

# CREATE TALBLE & INSERT DATA

In [13]:
spark.sql(f"""
    CREATE DATABASE IF NOT EXISTS {catalog}.{database}
""")

DataFrame[]

In [14]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {table_full_name} (
        order_id BIGINT,
        custormer_name STRING,
        order_date DATE
    ) USING iceberg
""")
spark.sql(f"""
    SELECT * FROM {table_full_name}
""").show()

+--------+--------------+----------+
|order_id|custormer_name|order_date|
+--------+--------------+----------+
+--------+--------------+----------+



In [15]:
spark.sql(f"""
    INSERT INTO {table_full_name} VALUES
    (1, 'Alice', DATE '2025-01-01'),
    (2, 'Bob', DATE '2025-01-02')
""")

DataFrame[]

In [16]:
spark.sql(f"""
    SELECT * FROM {table_full_name}
"""
).show()

+--------+--------------+----------+
|order_id|custormer_name|order_date|
+--------+--------------+----------+
|       1|         Alice|2025-01-01|
|       2|           Bob|2025-01-02|
+--------+--------------+----------+



## 첫 INSERT 후 data file(parquet) 이름과 파일 사이즈확인
- `00000-0-f9aba7dd-d97a-47a5-9568-2801d5586513-0-00001.parquet`(-> 998bytes)
- `00001-1-f9aba7dd-d97a-47a5-9568-2801d5586513-0-00001.parquet`(-> 984bytes)

In [17]:
spark.sql(f"""
    SELECT file_path,
        file_size_in_bytes
    FROM {table_full_name}.files
"""
).show(truncate=False)

+-----------------------------------------------------------------------------------------------------------+------------------+
|file_path                                                                                                  |file_size_in_bytes|
+-----------------------------------------------------------------------------------------------------------+------------------+
|s3://warehouse/my_database/customer_order/data/00000-0-16eb3d5d-793f-45d6-93ab-bb89c6eba637-0-00001.parquet|998               |
|s3://warehouse/my_database/customer_order/data/00001-1-16eb3d5d-793f-45d6-93ab-bb89c6eba637-0-00001.parquet|984               |
+-----------------------------------------------------------------------------------------------------------+------------------+



---

# ADD COLUMN(ALTER)
- Iceberg 는 ALTER .. ADD COLUMN 명령을 사용하여 컬럼을 추가하여도 데이터 파일에는 영향을 주지 않는다.
- 컬럼 추가 직후 metadata.json 파일과 data parquet 파일을 확인해보자.

In [11]:
spark.sql(f"""
    ALTER TABLE {table_full_name}
    ADD COLUMN total_amount DOUBLE
""")

DataFrame[]

- 아래 조회 결과와 같이 신규 추가된 컬럼은 NULL로 조회된다.

In [12]:
spark.sql(f"""
    SELECT * FROM {table_full_name}
"""
).show()

+--------+--------------+----------+------------+
|order_id|custormer_name|order_date|total_amount|
+--------+--------------+----------+------------+
|       1|         Alice|2025-01-01|        NULL|
|       2|           Bob|2025-01-02|        NULL|
+--------+--------------+----------+------------+



- 그러나 아래와 같이 데이터 파일에는 아무런 변화가 없다.
- `00000-0-f9aba7dd-d97a-47a5-9568-2801d5586513-0-00001.parquet`(-> 998bytes)
- `00001-1-f9aba7dd-d97a-47a5-9568-2801d5586513-0-00001.parquet`(-> 984bytes)

In [13]:
spark.sql(f"""
    SELECT file_path,
        file_size_in_bytes
    FROM {table_full_name}.files
"""
).show(truncate=False)

+-----------------------------------------------------------------------------------------------------------+------------------+
|file_path                                                                                                  |file_size_in_bytes|
+-----------------------------------------------------------------------------------------------------------+------------------+
|s3://warehouse/my_database/customer_order/data/00000-0-f9aba7dd-d97a-47a5-9568-2801d5586513-0-00001.parquet|998               |
|s3://warehouse/my_database/customer_order/data/00001-1-f9aba7dd-d97a-47a5-9568-2801d5586513-0-00001.parquet|984               |
+-----------------------------------------------------------------------------------------------------------+------------------+



- 이제 메타데이터 파일 내용을 살펴보자.
- 아래 메타데이터 포인터 조회 결과와 같이 현재 customer_order 테이블의 메타데이터 주소는 `00002-eb3870d1-8ce1-41f5-87b8-2e824c71f693.metadata.json` 을 바라보고 있다.

In [14]:
! curl -s http://iceberg-rest:8181/v1/namespaces/my_database/tables/customer_order

{"metadata-location":"s3://warehouse/my_database/customer_order/metadata/00002-eb3870d1-8ce1-41f5-87b8-2e824c71f693.metadata.json","metadata":{"format-version":2,"table-uuid":"90e64fb4-8441-45e5-ae93-106eb38b7f6d","location":"s3://warehouse/my_database/customer_order","last-sequence-number":1,"last-updated-ms":1760250382738,"last-column-id":4,"current-schema-id":1,"schemas":[{"type":"struct","schema-id":0,"fields":[{"id":1,"name":"order_id","required":false,"type":"long"},{"id":2,"name":"custormer_name","required":false,"type":"string"},{"id":3,"name":"order_date","required":false,"type":"date"}]},{"type":"struct","schema-id":1,"fields":[{"id":1,"name":"order_id","required":false,"type":"long"},{"id":2,"name":"custormer_name","required":false,"type":"string"},{"id":3,"name":"order_date","required":false,"type":"date"},{"id":4,"name":"total_amount","required":false,"type":"double"}]}],"default-spec-id":0,"partition-specs":[{"spec-id":0,"fields":[]}],"last-partition-id":999,"default-sort

- 아래 출력 결과와 같이 `schema-id` 값이 1인 fields 리스트 값을 확인해보면 컬럼 id 가 4인 `total_amount` 컬럼이 추가 되었다.
- 즉, `ALTER ADD COLUMN` 이후 데이터 파일에는 직접 영향을 주지 않고, 메타데이터만 갱신 되는 걸 알 수 있다.
- 다음 데이터 갱신 작업부터 data file 에 `total_amount` 컬럼에 value 가 추가되어 데이터가 적재된다.

In [15]:
metadata_file = "00002-eb3870d1-8ce1-41f5-87b8-2e824c71f693.metadata.json"
obj = get_metadata(file_name=metadata_file, table=table)
metadata_json = json.loads(obj["Body"].read().decode("utf-8"))

for key in IcebergMetadataJsonField:
    if key.value == 'schemas':
        print(f"🔑'KEY':{key.value}({FIELD_DESCRIPTIONS[key]})\n ▶︎'VALUE':{metadata_json[key.value]}")
        

print("\n📋 메타데이터 원본 JSON")
print(json.dumps(metadata_json, indent=2, ensure_ascii=False))

🔑'KEY':schemas(테이블에서 정의된 모든 스키마 리스트(컬럼 구조, 타입, 제약조건 등을 포함함))
 ▶︎'VALUE':[{'type': 'struct', 'schema-id': 0, 'fields': [{'id': 1, 'name': 'order_id', 'required': False, 'type': 'long'}, {'id': 2, 'name': 'custormer_name', 'required': False, 'type': 'string'}, {'id': 3, 'name': 'order_date', 'required': False, 'type': 'date'}]}, {'type': 'struct', 'schema-id': 1, 'fields': [{'id': 1, 'name': 'order_id', 'required': False, 'type': 'long'}, {'id': 2, 'name': 'custormer_name', 'required': False, 'type': 'string'}, {'id': 3, 'name': 'order_date', 'required': False, 'type': 'date'}, {'id': 4, 'name': 'total_amount', 'required': False, 'type': 'double'}]}]

📋 메타데이터 원본 JSON
{
  "format-version": 2,
  "table-uuid": "90e64fb4-8441-45e5-ae93-106eb38b7f6d",
  "location": "s3://warehouse/my_database/customer_order",
  "last-sequence-number": 1,
  "last-updated-ms": 1760250382738,
  "last-column-id": 4,
  "current-schema-id": 1,
  "schemas": [
    {
      "type": "struct",
      "schema-id": 0,
     

---

# ADD COLUMN 이후 INSERT DATA


In [17]:
spark.sql(f"""
    INSERT INTO {table_full_name} VALUES
    (3, 'Cryis', DATE '2025-01-03', 356.13),
    (4, 'Danger', DATE '2025-01-04', 800.43)
""").show()

++
||
++
++



In [19]:
spark.sql(f"""
    SELECT * FROM {table_full_name}
""").show()

+--------+--------------+----------+------------+
|order_id|custormer_name|order_date|total_amount|
+--------+--------------+----------+------------+
|       3|         Cryis|2025-01-03|      356.13|
|       4|        Danger|2025-01-04|      800.43|
|       1|         Alice|2025-01-01|        NULL|
|       2|           Bob|2025-01-02|        NULL|
+--------+--------------+----------+------------+



- ADD COLUMN 후 INSERT 하였을 때 2개의 parquet 파일이 추가 되었고, 이번에는 total_amount 에 대한 컬럼이 추가되어 file_size_in_bytes 값이 더 늘어난 것을 알 수 있다.

In [20]:
spark.sql(f"""
    SELECT file_path,
        file_size_in_bytes
    FROM {table_full_name}.files
"""
).show(truncate=False)

+-----------------------------------------------------------------------------------------------------------+------------------+
|file_path                                                                                                  |file_size_in_bytes|
+-----------------------------------------------------------------------------------------------------------+------------------+
|s3://warehouse/my_database/customer_order/data/00000-7-c4d4df9b-6243-4285-a2d4-7e306892b434-0-00001.parquet|1284              |
|s3://warehouse/my_database/customer_order/data/00001-8-c4d4df9b-6243-4285-a2d4-7e306892b434-0-00001.parquet|1291              |
|s3://warehouse/my_database/customer_order/data/00000-0-f9aba7dd-d97a-47a5-9568-2801d5586513-0-00001.parquet|998               |
|s3://warehouse/my_database/customer_order/data/00001-1-f9aba7dd-d97a-47a5-9568-2801d5586513-0-00001.parquet|984               |
+------------------------------------------------------------------------------------------------

In [18]:
parquet_add_column = "s3a://warehouse/my_database/customer_order/data/00000-0-16eb3d5d-793f-45d6-93ab-bb89c6eba637-0-00001.parquet"
df = spark.read.parquet(parquet_add_column)
df.show(truncate=False)

+--------+--------------+----------+
|order_id|custormer_name|order_date|
+--------+--------------+----------+
|1       |Alice         |2025-01-01|
+--------+--------------+----------+

